# Test MICOM alternatives  

In [1]:
from pathlib import Path
import sys
import cobra
import os
import pandas as pd
import pycomo
pycomo.configure_logger(level="info")
from cobra.io import read_sbml_model, write_sbml_model


2026-07-06 10:37:24,375 - PyCoMo - INFO - Logger initialized.


In [86]:
model_dir ="/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/add_exr"

In [78]:
com_fp = "/home/emma/Dokumente/thesis/communities"

In [79]:
HvSC1_l = ["1174_ex", "1234_ex", "1334_ex", "100_ex", "161_ex", "163_ex", "230_ex", "352_ex", "364_ex", "504_ex", "644_ex", "761_ex", "778_ex", "793_ex", "867_ex", "868_ex", "892_ex", "946_ex", "1018_ex", "1056_ex", "1124_ex", "1167_ex", "1391_ex", "1432_ex", "2774_ex", "2862_ex", "2872_ex"]
HvSC2_l = ["1174_ex", "1234_ex", "1334_ex", "262_ex", "397_ex", "428_ex", "459_ex", "638_ex", "709_ex", "790_ex", "796_ex", "895_ex", "939_ex", "947_ex", "978_ex", "997_ex", "1080_ex", "1101_ex", "1114_ex", "1208_ex", "1252_ex", "1338_ex", "1350_ex", "1357_ex", "1362_ex", "2751_ex"]

In [87]:
loaded_model_dict = {}
for file in os.listdir(model_dir):
    if not file.endswith(('.xml', '.sbml')):
        continue
        
    model = read_sbml_model(os.path.join(model_dir, file))
    model_id = model.id
    loaded_model_dict[model_id] = model

## SMETANA

In [27]:
HvSC1 = "Community_1"
HvSC2 = "Community_2"

data = []
for org1, org2 in zip(HvSC1_l, HvSC2_l):
    data.append({"organism id": org1, "community id": HvSC1})
    data.append({"organism id": org2, "community id": HvSC2})

communities_tsv = pd.DataFrame(data)

In [28]:
fp = os.path.join(com_fp, "communities_tsv.tsv")
communities_tsv.to_csv(fp, sep = "\t")

In [30]:
for model_id, model in loaded_model_dict.items():
    print(model_id, model.slim_optimize())

m_504 16.940837462507236
m_796 15.608400100671087
m_364 17.05601279883203
m_1018 16.911064811622026
m_1252 15.058534802380235
m_778 23.739399883164168
m_1234 16.61801880149043
m_1114 34.02455588233682
m_428 17.155128353212152
m_892 29.133080594698708
m_939 15.45016741434473
m_790 16.937279107816735
m_1124 16.97411924212672
m_895 16.722070507571996
m_230 16.97411924212672
m_1432 15.350758754809242
m_163 16.862736638015278
m_1056 14.868867996130303
m_161 16.97411924212672
m_1101 16.60627430599824
m_1338 16.758696817480196
m_1208 43.097770784266714
m_946 16.754780897143792
m_2862 16.940837462507236
m_397 16.937279107816735
m_352 16.06637030425786
m_459 17.173756045131135
m_262 24.655490576603714
m_638 43.097770784266714
m_100 15.45016741434473
m_997 16.937279107816735
m_1334 33.89239509835025
m_2751 17.184556315710317
m_2774 15.45016741434473
m_709 16.225924713218337
m_1391 16.97411924212672
m_2872 16.665506191918823
m_868 16.98723879551029
m_1357 33.18361439452638
m_761 16.62050183777037

## PyCoMo

In [88]:
def create_pycomo_commod(model_dir, model2com, community_name):
    named_models_com = {}
    single_org_models = []
    named_models = pycomo.load_named_models_from_dir(model_dir)
    for name, mod in named_models.items():
        if name in model2com:
            named_models_com[name] = mod
    for name, model in named_models_com.items():
        #print(name)
        single_org_model = pycomo.SingleOrganismModel(model, name)
        single_org_models.append(single_org_model)
    com_model_obj = pycomo.CommunityModel(single_org_models, community_name)
    return com_model_obj


#### create communities

In [111]:
HvSC1 = create_pycomo_commod(model_dir, HvSC1_l, "HvSC1")

2026-07-06 17:30:38,204 - PyCoMo - WARNING - Warning: model name 504_ex is not compliant with sbml id standards and was changed to _504_ex
2026-07-06 17:30:38,219 - PyCoMo - WARNING - Warning: model name 364_ex is not compliant with sbml id standards and was changed to _364_ex
2026-07-06 17:30:38,236 - PyCoMo - WARNING - Warning: model name 1018_ex is not compliant with sbml id standards and was changed to _1018_ex
2026-07-06 17:30:38,253 - PyCoMo - WARNING - Warning: model name 778_ex is not compliant with sbml id standards and was changed to _778_ex
2026-07-06 17:30:38,270 - PyCoMo - WARNING - Warning: model name 1234_ex is not compliant with sbml id standards and was changed to _1234_ex
2026-07-06 17:30:38,285 - PyCoMo - WARNING - Warning: model name 892_ex is not compliant with sbml id standards and was changed to _892_ex
2026-07-06 17:30:38,307 - PyCoMo - WARNING - Warning: model name 1124_ex is not compliant with sbml id standards and was changed to _1124_ex
2026-07-06 17:30:38,3

In [116]:
sum_sc1 = HvSC1.summary()

In [125]:
# Look for the biomass reaction fluxes for your members
solution = HvSC1.run_fba()
for rxn in HvSC1._f_reactions:
    if "biomass" in rxn.id.lower():
        print(f"Reaction: {rxn.id} | Growth Rate: {solution.flux[rxn.id]:.4f}")

Reaction: SK__504_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__364_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__1018_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__778_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__1234_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__892_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__1124_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__230_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__1432_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__163_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__1056_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__161_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__946_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__2862_ex_to_community_biomass_ub | Growth Rate: 0.3648
Reaction: SK__352_ex_to_community_biomass_ub | Growth Ra

In [124]:
solution

,reaction_id,flux
_504_ex_TF_3ump__504_ex_e,_504_ex_TF_3ump__504_ex_e,-0.064847
_504_ex_TF_6pgc__504_ex_e,_504_ex_TF_6pgc__504_ex_e,-0.021694
_504_ex_TF_LalaDgluMdap__504_ex_e,_504_ex_TF_LalaDgluMdap__504_ex_e,-0.015492
_504_ex_TF_R_3httdca__504_ex_e,_504_ex_TF_R_3httdca__504_ex_e,-0.043387
_504_ex_TF_ac__504_ex_e,_504_ex_TF_ac__504_ex_e,0.001696
...,...,...
SK__1174_ex_FRUpts_ub,SK__1174_ex_FRUpts_ub,0.367676
SK__1174_ex_to_community_biomass_ub,SK__1174_ex_to_community_biomass_ub,0.364795
f_final,f_final,1.000000
abundance_reaction,abundance_reaction,15.052613


In [113]:
HvSC1.convert_to_fixed_abundance()
abundance_sc1 = HvSC1.generate_equal_abundance_dict()
HvSC1.apply_fixed_abundance(abundance_sc1)

In [93]:
uptake_sc1 = HvSC1.summary().uptake_flux

In [94]:
len(uptake_sc1)

107

In [95]:
uptake_sc1 = uptake_sc1.sort_values("flux", ascending=False)
uptake_sc1.head(20)

,flux,reaction,metabolite
EX_o2_medium,333.479715,EX_o2_medium,o2_medium
EX_h2_medium,186.948158,EX_h2_medium,h2_medium
EX_h_medium,148.713312,EX_h_medium,h_medium
EX_for_medium,129.500811,EX_for_medium,for_medium
EX_meoh_medium,110.939442,EX_meoh_medium,meoh_medium
EX_pea_medium,32.729422,EX_pea_medium,pea_medium
EX_f6p_medium,32.617525,EX_f6p_medium,f6p_medium
EX_no3_medium,20.009243,EX_no3_medium,no3_medium
EX_val__L_medium,6.372716,EX_val__L_medium,val__L_medium
EX_ser__L_medium,5.605689,EX_ser__L_medium,ser__L_medium


In [96]:
uptake_sc1 = HvSC1.summary().uptake_flux

In [97]:
HvSC2 = create_pycomo_commod(model_dir, HvSC2_l, "HvSC2")

2026-07-06 13:43:49,658 - PyCoMo - WARNING - Warning: model name 796_ex is not compliant with sbml id standards and was changed to _796_ex
2026-07-06 13:43:49,674 - PyCoMo - WARNING - Warning: model name 1252_ex is not compliant with sbml id standards and was changed to _1252_ex
2026-07-06 13:43:49,690 - PyCoMo - WARNING - Warning: model name 1234_ex is not compliant with sbml id standards and was changed to _1234_ex
2026-07-06 13:43:49,704 - PyCoMo - WARNING - Warning: model name 1114_ex is not compliant with sbml id standards and was changed to _1114_ex
2026-07-06 13:43:49,720 - PyCoMo - WARNING - Warning: model name 428_ex is not compliant with sbml id standards and was changed to _428_ex
2026-07-06 13:43:49,734 - PyCoMo - WARNING - Warning: model name 939_ex is not compliant with sbml id standards and was changed to _939_ex
2026-07-06 13:43:49,748 - PyCoMo - WARNING - Warning: model name 790_ex is not compliant with sbml id standards and was changed to _790_ex
2026-07-06 13:43:49,7

In [98]:
HvSC2.convert_to_fixed_abundance()
abundance_sc2 = HvSC2.generate_equal_abundance_dict()
HvSC2.apply_fixed_abundance(abundance_sc2)

2026-07-06 13:46:03,664 - PyCoMo - INFO - No community model generated yet. Generating now:
2026-07-06 13:46:03,697 - PyCoMo - INFO - Identified biomass reaction from objective: Growth
2026-07-06 13:46:03,697 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-07-06 13:46:04,011 - PyCoMo - INFO - Identified biomass reaction from objective: Growth
2026-07-06 13:46:04,011 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-07-06 13:46:04,548 - PyCoMo - INFO - Identified biomass reaction from objective: Growth
2026-07-06 13:46:04,549 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-07-06 13:46:05,123 - PyCoMo - INFO - Identified biomass reaction from objective: Growth
2026-07-06 13:46:05,123 - PyCoMo - INFO - Note: no products in the objective function, adding biomass to it.
2026-07-06 13:46:05,602 - PyCoMo - INFO - Identified biomass reaction from objective: Growth
2026-07

In [99]:
uptake_sc2 = HvSC2.summary().uptake_flux

#### complete medium with PyCoMo

In [107]:
base_medium = pd.read_csv("/home/emma/Dokumente/thesis/media_creation/created_media/combined_af.csv", header = None)
base_medium = base_medium.dropna()
base_medium = dict(zip(base_medium[0], base_medium[1]))
base_medium = {
    (k[:-2] + "_medium" if k.endswith("_m") else k): v 
    for k, v in base_medium.items()
}


In [109]:
cleaned_medium = {
    rxn_id: bound 
    for rxn_id, bound in base_medium.items() 
    if rxn_id in uptake_sc2["reaction"]
}

print(f"Kept {len(cleaned_medium)} out of {len(base_medium)} components.")

Kept 78 out of 98 components.


## REMIND

In [110]:
import os
import glob
import os
import time
from sys import argv
from pytfa.optim.utils import symbol_sum

import numpy as np
import pandas as pd
from pytfa.io.json import load_json_model
from pytfa.optim.constraints import ModelConstraint

from remind.core.exchanges import imeanalysis_all_size
from remind.core.medium import constrain_sources
from remind.core.parsimonious import *


from os.path import join
import cobra
from pathlib import Path


ImportError: cannot import name 'LinearizationConstraint' from 'pytfa.optim.constraints' (/home/emma/miniconda3/envs/community_modeling/lib/python3.10/site-packages/pytfa/optim/constraints.py)

In [ ]:
def get_base_path():
    """
    Returns the base path of the installed remind package if importable.
    Otherwise returns an empty string (or None).
    """
    try:
        import remind
        base = Path(remind.__file__).resolve().parent
        return str(base.parents[1])
    except ImportError:
        # Not installed (e.g., inside docker where local tree is used)
        return ""   # or return None
base=get_base_path()


In [ ]:

#these code is the one that were used for the latest analysis

_, model_num,tol=argv
model_no=int(model_num)

max_alternative=10


tolerance_lb=1.0/float(tol)
tolerance=tolerance_lb*1.1 #a bit of gap given


growth_limit=0.2
biomass='Growth'
allowed=True
max_consmp=3 #maxmum carbon sources to be used at the same time from the given list
max_flux_lim=1.0
constrain_fluxes=True
#todo put imm and iee scripts in other folders

CPLEX = 'optlang-cplex'
GUROBI = 'optlang-gurobi'
GLPK = 'optlang-glpk'
solver = CPLEX


t=time.time()

#TODO check_here
# parsimonious='all_uptakes_mw'
#to make the yield_constraint with respect to the carbon uptake
parsimonious='c_uptakes_only_C_moles' #yield is calc
#parsimonious='c_uptakes'



path=base+'/remind/models/bee_models/core_members/tfa_real_101023' #for the two model dimes
path=base+'/remind/models/bee_models/core_members/tfa_real_101023'
allFiles = glob.glob(path + "/*"+"json")
biomass_rxn='Growth'

# model = load_json_model_tmodel_struct(allFiles[model_no])
model = load_json_model(allFiles[model_no])


print('Number of reactions {} and number of genes {} for model {}'.format(len(model.reactions),len(model.genes),model.id))
print('Number of metabolites {}'.format(len(model.metabolites)))


allFolders = os.listdir(path)
model_name=model.name.split('tfa_')[-1]
native_exchanges=len(model.exchanges)
#read experimental data checked for feasibility
#all have only 1 alternative apart from kullabergensis
data_file=base+'/remind/projects/bee_project/stats/'

data_dir_cobra=base+'/remind/models/bee_models/core_members/carveme_models'
model_cobra=cobra.io.read_sbml_model(join(data_dir_cobra,"{}.xml".format(model_name)))
# model_name=allFolders[model_no].split(".xml")[0]
# t=time.time()


"possible carbon sources data"
frame_possible=pd.read_csv(data_file+"possible_c_sources_list_300822_initial.csv",index_col=None) #main carbon source


"union of essential metabolites data"
frame_union1=pd.read_hdf(base+'/remind/projects/bee_project/constrained_medium/essential_mets/essential_mets_w_cat_repr_updated_010922_TFA.h5')
frame_union2=pd.read_hdf(base+'/remind/projects/bee_project/constrained_medium/essential_mets/essential_mets_w_cat_repr_updated_061023_2_TFA.h5')
frame_union2_species = frame_union2[frame_union2.model.isin([model_name])]
frame_union= frame_union1.append(frame_union2_species).reset_index()

# frame_union= frame_union1

"if we only want to add the essential ones for each model"
if allowed:
    met_pool = list(frame_union.metabolites.unique())

else:
    frame_met=frame_union[frame_union.model.isin([model_name])]
    met_pool = list(frame_met.metabolites.unique())

# met_pool=list(frame_union.metabolites.unique())
#for all
met_pool=[k for k in met_pool if k!='EX_coa_e']

# list_all=[k for k in frame_screened.mets.unique()]
list_possible=[k for k in frame_possible.annotation_bigg.unique() if k!='no']

# list_sink=[k for k in model.reactions if "sink" in k.id]

diff_rxns=[k for k in model.reactions if k not in model_cobra.reactions]
sink_diff=[k for k in diff_rxns if "sink" in k.id]

model.remove_reactions(sink_diff) #remove the sink reactions that were added when checking for BBB production
model.repair()



c_sources_to_include=list(set(list_possible+met_pool))

# "only for gapicola make this"
#activate this anyways
"""this was for fdp now with thermo firsttry without fdp"""
if model_name=='Gilliamella_apicola_wkB1GF':
    frame_fdp=pd.read_hdf(base+'/remind/projects/bee_project/constrained_medium/fdp_stats/all_fdps_for_bee_gapicola_050822.h5')
    list_fdp=[]
    for fdp in frame_fdp.fdp_list:
        list_fdp+=fdp
    list_fdp=list(set(list_fdp))
    chosen=['EX_23camp_e', 'EX_cmp_e', 'EX_lac__D_e', 'EX_nac_e','EX_etoh_e']
    remove_gapic=[k for k in list_fdp if k not in chosen]
    #latest removal because they alternate completely based on pre runs
    remove_gapic_2=[ 'EX_csn_e', 'EX_ura_e', 'EX_cytd_e', 'EX_dad_2_e','EX_mmet_e']
    remove_gapic=['EX_nac_e','EX_gln__L_e']+remove_gapic_2
    c_sources_to_include=[k for k in c_sources_to_include if k not in remove_gapic]


if model_name=="Snodgrassella_alvi_wkB2GF":
    not_accept_list=["EX_etoh_e","EX_meoh_e","EX_hxan_e"]
    c_sources_to_include = [k for k in c_sources_to_include if k  not in not_accept_list]


if model_name=='Gilliamella_apicola_wkB1GF':
    not_accept_list=['EX_4hthr_e','EX_pheme_e','EX_etoh_e','EX_uri_e']
    c_sources_to_include = [k for k in c_sources_to_include if k not in not_accept_list]



"remove them"
list_remove=[k for k in listcarbonsources(model) if k not in c_sources_to_include]

#this is to force o2 uptake as Snodgrasella is modeled as aerobic
if model_name=="Snodgrassella_alvi_wkB2GF":
    model.reactions.EX_o2_e.bounds=(-25,-1)

rxns_to_rmv=list_remove

model.remove_reactions(rxns_to_rmv)
after_exchanges=len(model.exchanges)

list_not_main_c_source=[model.reactions.get_by_id(k) for k in listcarbonsources(model) if k in met_pool]

print("For model {} native exchanges is {} after screening is {}".format(model.name,native_exchanges,after_exchanges))


max_flux=25

if constrain_fluxes:
    for k in model.exchanges:
        if k.id!='EX_o2_e':
            k.bounds=(-25,max_flux)
    for r in  list_not_main_c_source:
        r.bounds=(-1*max_flux_lim,max_flux)

#new for inorganics
list_carbon_sources=listcarbonsources(model)
inorganics=[k for k in model.exchanges if k.id not in list_carbon_sources]
inorganics=[k for k in inorganics if k.id!="EX_o2_e"]

for rxn_inor in inorganics:
    rxn_inor.bounds=(-25,50)

if 'EX_glc__D_e' in model.reactions:
    model.reactions.get_by_id('EX_glc__D_e').upper_bound=0

if 'EX_fru_e' in model.reactions:
    model.reactions.get_by_id('EX_fru_e').upper_bound=0

if 'EX_cit_e' in model.reactions:
    model.reactions.get_by_id('EX_cit_e').upper_bound=0

if 'EX_icit_e' in model.reactions:
    model.reactions.get_by_id('EX_icit_e').upper_bound=0

#new additional info
"it is known that it cannot catabolize carbohydrates"
if model_name=="Snodgrassella_alvi_wkB2GF":
    model.reactions.get_by_id('EX_fru_e').lower_bound = 0
    model.reactions.get_by_id('EX_glc__D_e').lower_bound = 0


# Solver settings
def apply_solver_settings(model, solver = solver):
    model.solver = solver
    # model.solver.configuration.verbosity = 1
    model.solver.problem.parameters.reset()
    model.solver.configuration.tolerances.feasibility = 1e-9
    model.solver.configuration.tolerances.optimality = 1e-9
    model.solver.configuration.tolerances.integrality = 1e-9

    if solver == 'optlang_gurobi':
        model.solver.problem.Params.NumericFocus = 3
    model.solver.configuration.presolve = True

apply_solver_settings(model)


limit=growth_limit
model.reactions.get_by_id(biomass).upper_bound=limit
model.reactions.get_by_id(biomass).lower_bound=limit*0.98

#force citrate for snod


"""catabolite repression decide to put 0 or 1"""
list_carbon_sources = listcarbonsources(model)
list_catabolite=[k for k in list_possible if k in list_carbon_sources ]
constrain_sources(model,list_catabolite, max_cnsmp=max_consmp, min_cnsmp=0)


#for sink reaction minimization


#
'2) get list of rxn names for parsimonious uptakes'
if (parsimonious=='c_uptakes'):
    list_carbon_sources=listcarbonsources(model)
    '3)apply parsimonious and get solution and expression'
    'minimize uptake of carbon sources or all uptakes'
    expr_pars,sol_pars=minimizecarbonuptakes(model, list_carbon_sources)
 #todo write another function for C molar balance
if (parsimonious=='c_uptakes_only_C_moles'):
    list_carbon_sources=listcarbonsources(model)
    '3)apply parsimonious and get solution and expression'
    'minimize uptake of carbon sources or all uptakes'
    expr_pars,sol_pars=minimizecarbonuptakes_cmoles_only(model, list_carbon_sources)
if (parsimonious=='c_uptakes_mw'):
    list_carbon_sources=listcarbonsources(model)
    '3)apply parsimonious and get solution and expression'
    'minimize uptake of carbon sources or all uptakes'
    expr_pars,sol_pars=minimizecarbonuptakes_by_weight(model, list_carbon_sources,molecular_weight='formula_weight')
if (parsimonious=='all_uptakes_mw'):
    #list_carbon_sources=listcarbonsources(mytfa)
    '3)apply parsimonious and get solution and expression'
    'minimize uptake of carbon sources or all uptakes'
    # rxn_list=[model.reactions.get_by_id(r) for r in listcarbonsources(model) if r in list_possible]
    expr_pars,sol_pars=minimizealluptakes_by_weight(model,model.exchanges, molecular_weight='formula_weight')
elif (parsimonious=='all_uptakes'):
    expr_pars,sol_pars=minimizealluptakes(model)

print(sol_pars)
'4)add constrain for parsimonious'
'minimize uptake of carbon sources or all uptakes user defined'
cons0 = model.add_constraint(ModelConstraint,
                              model,
                              expr_pars,
                              id_='pars_constraint',
                                #fix uptake
                              lb=(sol_pars.objective_value)*tolerance_lb,
                              ub=(sol_pars.objective_value)*(tolerance))


model.repair()



rxns_to_consider=[model.reactions.get_by_id(k) for k in list_carbon_sources ]


'generate untill alternatives until the given condition'

my_list,df_all=imeanalysis_all_size(model,rxns_to_consider,getAlternatives=True,max_alternative=max_alternative,biomass=biomass,stop_cond=True,criteria=2)

'generate untill max alternative'
#my_list,df_all=imeanalysis_all_size_o2(mytfa2,mytfa2.exchanges,getAlternatives=True,max_alternative=10)

frame = pd.concat(my_list, ignore_index=True)
# frame['uptake_min']=np.ones(frame.shape[0])*sol_pars.objective_value
frame['yield_perc']=np.ones(frame.shape[0])*1.0/tolerance_lb

alternative=frame.alternative.max()
elapsed = time.time() - t
print('time for ime analysis', elapsed)

from remind.utils.postprocessing import *
add_size(frame,['alternative'],'alt_size')

# from remind.core.decomposition import find_essential_alternates_from_dime_list
# a,b=find_essential_alternates_from_dime_list(frame)


#output file to save the results
output_file_all=base+'/remind/projects/tutorial/tutorial_bee_DiMEs_{}_limited_carbon/{}/all_vars'.format(allowed,model.name.split('tfa_')[-1])
if not os.path.exists(output_file_all):
    os.makedirs(output_file_all)

output_file_alts=base+'/remind/projects/tutorial/tutorial_bee_DiMEs_{}_limited_carbon/{}/alternatives'.format(allowed,model.name.split('tfa_')[-1])
if not os.path.exists(output_file_alts):
    os.makedirs(output_file_alts)

#"store them in hdf5 files"
frame.to_hdf(output_file_alts+'/alternative_mets_for_{}_growth_{}_alt_{}_pars_{}_tol_pars_{}_tol_lb_{}.h5'.format(model.id,limit,alternative,parsimonious,tolerance,tolerance_lb),key='s')
df_all.to_hdf(output_file_all+'/alternatives_all_vars_for_{}_growth_{}_alt_{}_pars_{}_tol_pars_{}_tol_lb_{}.h5'.format(model.id,limit,alternative,parsimonious,tolerance,tolerance_lb),key='s')


## test gapseq models

In [2]:
from cobra.io import read_sbml_model

In [3]:
m100 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/gapseq_models/100-draft.xml")

'' is not a valid SBML 'SId'.


In [6]:
m100 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/Draft_models/100_faa.xml")

Adding exchange reaction EX_14glucan_e with default bounds for boundary metabolite: 14glucan_e.
Adding exchange reaction EX_15dap_e with default bounds for boundary metabolite: 15dap_e.
Adding exchange reaction EX_2m35mdntha_e with default bounds for boundary metabolite: 2m35mdntha_e.
Adding exchange reaction EX_2mbald_e with default bounds for boundary metabolite: 2mbald_e.
Adding exchange reaction EX_35dnta_e with default bounds for boundary metabolite: 35dnta_e.
Adding exchange reaction EX_3mb_e with default bounds for boundary metabolite: 3mb_e.
Adding exchange reaction EX_4abut_e with default bounds for boundary metabolite: 4abut_e.
Adding exchange reaction EX_4ahmmp_e with default bounds for boundary metabolite: 4ahmmp_e.
Adding exchange reaction EX_4hba_e with default bounds for boundary metabolite: 4hba_e.
Adding exchange reaction EX_4hbald_e with default bounds for boundary metabolite: 4hbald_e.
Adding exchange reaction EX_4hbz_e with default bounds for boundary metabolite: 4h

In [10]:
m100.metabolites.glc__D_c

Metabolite identifier,glc__D_c
Name,D-Glucose
Memory address,0x77bb85dd89a0
Formula,C6H12O6
Compartment,C_c
In 25 reaction(s),"GALS3, MPL, G6PP, TREH, G1PP, GALM1, SUCR, HEX1, MLTG4, LMN2, MLTG2, XYLI2, MLTG1, BGLA, MLTG3, GLCt2, LACZ, MLTG6, GLS, MALT, MLTG5, GalMr_2, TRE6PH, GLS2, GLCt2pp"


In [59]:
m1252 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/gapseq_models/1252-draft.xml")

'' is not a valid SBML 'SId'.


In [61]:
m364 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/gapseq_models/364-draft.xml")

'' is not a valid SBML 'SId'.


In [66]:
m1334 = read_sbml_model("/home/emma/Dokumente/thesis/Model_generation_curation/gapseq_models/1334-draft.xml")

'' is not a valid SBML 'SId'.


In [67]:
m1334.metabolites.cpd00082

AttributeError: DictList has no attribute or entry cpd00082

In [73]:
for r in m1252.metabolites.cpd00054_c0.reactions:
    print(r)

rxn06446_c0: cpd00002_c0 + cpd00054_c0 + cpd11921_c0 <=> cpd00012_c0 + cpd00018_c0 + 2.0 cpd00067_c0 + cpd12132_c0
rxn00692_c0: cpd00001_c0 + cpd00033_c0 + cpd00125_c0 <=> cpd00054_c0 + cpd00087_c0
rxn15964_c0: cpd00054_c0 + cpd00895_c0 --> cpd00001_c0 + cpd00033_c0 + cpd02679_c0
bio1: 34.922907081471 cpd00001_c0 + 40.1654763175579 cpd00002_c0 + 0.00338811164817026 cpd00003_c0 + 0.00338811164817026 cpd00006_c0 + 0.00338811164817026 cpd00010_c0 + 0.00338811164817026 cpd00015_c0 + 0.00338811164817026 cpd00016_c0 + 0.00338811164817026 cpd00017_c0 + 0.249807760211032 cpd00023_c0 + 0.00338811164817026 cpd00028_c0 + 0.00729445361450978 cpd00030_c0 + 0.581552465771283 cpd00033_c0 + 0.00729445361450978 cpd00034_c0 + 0.487624747931933 cpd00035_c0 + 0.203371657456662 cpd00038_c0 + 0.325749319315185 cpd00039_c0 + 0.228823908353305 cpd00041_c0 + 0.00338811164817026 cpd00042_c0 + 0.00729445361450978 cpd00048_c0 + 0.2807839224772 cpd00051_c0 + 0.126317799662523 cpd00052_c0 + 0.249807760211032 cpd000

In [51]:
m100.reactions.EX_cpd00027_e0

Reaction identifier,EX_cpd00027_e0
Name,D-Glucose-e0 Exchange
Memory address,0x70a45d7c45b0
Stoichiometry,cpd00027_e0 --> D-Glucose-e0 -->
GPR,
Lower bound,0.0
Upper bound,1000.0
